In [ ]:
# -*- coding: utf-8 -*-
# Licensed under a 3-clause BSD style license - see LICENSE.rst
import logging
import os
import warnings
from collections import OrderedDict
from multiprocessing import Pool

import astropy
from astropy.table import Table, hstack
import astropy.units as u
from astropy.io import ascii
from astropy.constants import alpha, c, e, hbar, m_e, m_p, sigma_sb
from astropy.utils.data import get_pkg_data_filename
from astropy.cosmology import WMAP9 as cosmo

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
import time
from numba import njit,prange

from Validator import (
    validate_array,
    validate_physical_type,
    validate_scalar,
)
#from .model_utils import memoize
from Utils import trapz_loglog,trapz_loglog_nd

import Models
import RadiativeNEW
import grbloader
from GRBmodel import GRBModel1
from GRBmodelstr import GRBModel_topstruc
from Models import EblAbsorptionModel

from astropy.units import def_physical_type

try:
    #def_physical_type(u.Unit("1 / eV"), "differential energy")
    def_physical_type(u.erg / u.cm**2 / u.s, "flux")
    def_physical_type(u.Unit("1/(s cm2 erg)"), "differential flux")
    def_physical_type(u.Unit("1/(s erg)"), "differential power")
    def_physical_type(u.Unit("1/TeV"), "differential energy")
    def_physical_type(u.Unit("1/cm3"), "number density")
    def_physical_type(u.Unit("1/(eV cm3)"), "differential number density")

except ValueError:
    print("New quantities already defined")


import astropy.constants as con
from astropy.constants import alpha, c, e, hbar, m_e, m_p, sigma_sb
m_e = con.m_e.cgs.value
c = con.c.cgs.value
mec2_eV = (con.m_e * con.c ** 2.).to('eV').value
h = con.h.cgs.value
el = con.e.gauss.value
erg_to_eV = 624150912588.3258  # conversion from erg to eV
sigma_T = con.sigma_T.cgs.value
mpc2 = (con.m_p * con.c ** 2.).to('eV')
mpc2_erg = mpc2.to('erg').value

from astropy import units as u
from astropy.constants import m_e, c

mec2 = (m_e * c**2).cgs
mec2_unit = u.Unit(mec2)



New quantities already defined


In [ ]:
from matplotlib import cm
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.colorbar import ColorbarBase

def truncate_colormap(cmap, minval=0.0, maxval=1.0, n=100):
    new_cmap = LinearSegmentedColormap.from_list(
        f'trunc({cmap.name},{minval:.2f},{maxval:.2f})',
        cmap(np.linspace(minval, maxval, n))
    )
    return new_cmap

vmin = 0.1
vmax = 1
original_cmap = plt.cm.plasma

cmap1 = truncate_colormap(original_cmap, vmin, vmax)
fig, ax = plt.subplots(figsize=(10, 1))
fig.subplots_adjust(bottom=0.5)
norm = Normalize(vmin, vmax)  # da 0 a 1 perché stai solo mostrando il segmento
cb = ColorbarBase(ax,cmap=cmap1,norm=norm,orientation='horizontal')
cb.set_label(f'Segmento di viridis da {vmin} a {vmax}')
plt.show()

vmin = 0.1
vmax = 0.9
original_cmap2 = plt.cm.viridis

cmap2 = truncate_colormap(original_cmap2, vmin, vmax)
fig, ax = plt.subplots(figsize=(10, 1))
fig.subplots_adjust(bottom=0.5)
norm = Normalize(vmin, vmax)  # da 0 a 1 perché stai solo mostrando il segmento
cb = ColorbarBase(ax,cmap=cmap2,norm=norm,orientation='horizontal')
cb.set_label(f'Segmento di viridis da {vmin} a {vmax}')
plt.show()

In [2]:
import time
from scipy.integrate import simps, quad, dblquad

@njit
def trapz_loglog_nd(y, x):
    out_shape = y.shape[:-1]
    res = np.zeros(out_shape)

    n = x.shape[0]
    for idx in np.ndindex(out_shape):
        s = 0.0
        for i in range(n - 1):
            x1 = x[i]; x2 = x[i+1]
            y1 = y[idx + (i,)]; y2 = y[idx + (i+1,)]
            if y1 <= 0.0 or y2 <= 0.0 or x1 <= 0.0 or x2 <= 0.0:
                continue
            b = np.log(y2 / y1) / np.log(x2 / x1)
            if np.abs(b + 1.0) > 1e-10:
                s += (y1 * (x2 * (x2 / x1)**b - x1)) / (b + 1.0)
            else:
                s += x1 * y1 * np.log(x2 / x1)
        res[idx] = s

    return res

@njit
def trapz_loglog_nd_fast(y, x):

    out_shape = y.shape[:-1]
    n = x.shape[0]
    res = np.zeros(out_shape)

    # 🔹 Precalcola i logaritmi di x per evitare ripetizioni costose
    logx = np.log(x)

    # 🔹 Loop su tutti gli indici tranne l'ultimo asse
    for idx in np.ndindex(out_shape):
        s = 0.0
        for i in range(n - 1):
            x1 = x[i]; x2 = x[i+1]
            y1 = y[idx + (i,)]; y2 = y[idx + (i+1,)]

            # Salta intervalli non validi
            if y1 <= 0.0 or y2 <= 0.0 or x1 <= 0.0 or x2 <= 0.0:
                continue

            # 🔹 Calcola b in modo stabile
            b = np.log(y2 / y1) / (logx[i+1] - logx[i])

            # 🔹 Versione senza 'if': aggiungi piccolo offset per evitare divisione per zero
            denom = b + 1.0 + 1e-14
            s += (y1 * (x2 * (x2 / x1)**b - x1)) / denom
        res[idx] = s
    return res


@njit
def trapz_numba(y, x):
    """
    Trapezoidal integration along the last axis.
    y : 2D array
    x : 1D array (same length as last axis of y)
    """
    n = x.shape[0]
    out_shape = y.shape[:-1]
    out = np.zeros(out_shape)

    # Scorri tutti gli indici tranne l’ultimo asse
    for idx in np.ndindex(out_shape):
        s = 0.0
        for i in range(n - 1):
            s += 0.5 * (y[idx + (i+1,)] + y[idx + (i,)]) * (x[i+1] - x[i])
        out[idx] = s

    return out

@njit
def simps_numba(y, x,step=1):
    """
    Integrazione con la regola di Simpson lungo l'ultima dimensione di y.

    Parametri
    ----------
    y : array (..., N)
        Valori della funzione da integrare.
    x : array (N,)
        Ascisse corrispondenti.
    Ritorna
    -------
    res : array (...,)
        Integrale lungo l'ultima dimensione.
    """

        # 🔹 sottocampionamento
    x = x[::step]
    y_sub = y[..., ::step]

    out_shape = y_sub.shape[:-1]
    res = np.zeros(out_shape)

    #out_shape = y.shape[:-1]
    #res = np.zeros(out_shape)

    n = x.shape[0]
    if n < 3:
        # fallback per troppo pochi punti
        for idx in np.ndindex(out_shape):
            res[idx] = 0.0
        return res

    # Se pari, usa n-1 punti per avere numero dispari
    if n % 2 == 0:
        n -= 1

    h = (x[n-1] - x[0]) / (n - 1)

    for idx in np.ndindex(out_shape):
        s = y[idx + (0,)] + y[idx + (n-1,)]
        for i in range(1, n-1, 2):
            s += 4.0 * y[idx + (i,)]
        for i in range(2, n-2, 2):
            s += 2.0 * y[idx + (i,)]
        res[idx] = s * h / 3.0

    return res


@njit
def simps_numba_uniform(y, x):
    """
    Regola di Simpson per griglie uniformi (corretta anche per N pari).
    """
    n = x.shape[0]
    h = (x[-1] - x[0]) / (n - 1)

    out_shape = y.shape[:-1]
    res = np.zeros(out_shape)

    for idx in np.ndindex(out_shape):
        # se n è pari → usa Simpson 3/8 sugli ultimi 4 punti
        if n % 2 == 0:
            # togli l’ultimo intervallo dalla parte principale
            n_main = n - 3
            s = y[idx + (0,)] + y[idx + (n_main-1,)]
            for i in range(1, n_main-1, 2):
                s += 4.0 * y[idx + (i,)]
            for i in range(2, n_main-1, 2):
                s += 2.0 * y[idx + (i,)]
            I = s * h / 3.0

            # aggiungi Simpson 3/8 sugli ultimi 3 intervalli
            I += (3*h/8.0) * (y[idx + (n-4,)] + 3*y[idx + (n-3,)] +
                            3*y[idx + (n-2,)] + y[idx + (n-1,)])
        else:
            # normale Simpson 1/3
            s = y[idx + (0,)] + y[idx + (n-1,)]
            for i in range(1, n-1, 2):
                s += 4.0 * y[idx + (i,)]
            for i in range(2, n-1, 2):
                s += 2.0 * y[idx + (i,)]
            
            I = s * h / 3.0

        res[idx] = I

    return res

from numba import njit
import numpy as np

@njit
def simps_numba_uniform_kahan(y, x):
    """
    Regola di Simpson per griglie uniformi (con correzione per N pari e somma compensata di Kahan)
    """
    n = x.shape[0]
    h = (x[-1] - x[0]) / (n - 1)

    out_shape = y.shape[:-1]
    res = np.zeros(out_shape)

    for idx in np.ndindex(out_shape):
        # --- inizializzazione ---
        s = y[idx + (0,)] + y[idx + (n-1,)]

        # 🔹 Somma compensata (Kahan) per i termini con peso 4
        c = 0.0
        for i in range(1, n-1, 2):
            y_i = 4.0 * y[idx + (i,)]
            y_corr = y_i - c
            t = s + y_corr
            c = (t - s) - y_corr
            s = t

        # 🔹 Somma compensata per i termini con peso 2
        for i in range(2, n-1, 2):
            y_i = 2.0 * y[idx + (i,)]
            y_corr = y_i - c
            t = s + y_corr
            c = (t - s) - y_corr
            s = t

        I = s * h / 3.0

        # 🔹 Correzione bordo per N pari (formula "5-8-1" come SciPy)
        if n % 2 == 0:
            I += (h / 12.0) * (5*y[idx + (n-1,)] + 8*y[idx + (n-2,)] - y[idx + (n-3,)])


        res[idx] = I

    return res


@njit
def simps_logspace_numba2(y, x):
    """
    Regola di Simpson per griglie uniformi in log10(x),
    corretta anche per N pari (usa 3/8 sugli ultimi 3 intervalli).
    """
    n = x.shape[0]
    # coordinate logaritmiche
    t0 = np.log10(x[0])
    t1 = np.log10(x[-1])
    h = (t1 - t0) / (n - 1)

    out_shape = y.shape[:-1]
    res = np.zeros(out_shape)

    for idx in np.ndindex(out_shape):
        if n < 3:
            res[idx] = 0.0
            continue

        # funzione trasformata: f_i = y_i * x_i * ln(10)
        # perché dx = x * ln(10) * dt
        if n % 2 == 0:
            # n pari → 1/3 fino a n-4, poi 3/8 finale
            n_main = n - 3
            s = y[idx + (0,)] * x[0] * np.log(10.0) + y[idx + (n_main-1,)] * x[n_main-1] * np.log(10.0)
            for i in range(1, n_main-1, 2):
                s += 4.0 * y[idx + (i,)] * x[i] * np.log(10.0)
            for i in range(2, n_main-1, 2):
                s += 2.0 * y[idx + (i,)] * x[i] * np.log(10.0)
            I = s * h / 3.0

            # Simpson 3/8 sugli ultimi 4 punti
            I += (3*h/8.0) * (
                y[idx + (n-4,)] * x[n-4] * np.log(10.0)
                + 3.0 * y[idx + (n-3,)] * x[n-3] * np.log(10.0)
                + 3.0 * y[idx + (n-2,)] * x[n-2] * np.log(10.0)
                + y[idx + (n-1,)] * x[n-1] * np.log(10.0)
            )

        else:
            # n dispari → normale Simpson 1/3 su tutti i punti
            s = y[idx + (0,)] * x[0] * np.log(10.0) + y[idx + (n-1,)] * x[n-1] * np.log(10.0)
            for i in range(1, n-1, 2):
                s += 4.0 * y[idx + (i,)] * x[i] * np.log(10.0)
            for i in range(2, n-1, 2):
                s += 2.0 * y[idx + (i,)] * x[i] * np.log(10.0)
            I = s * h / 3.0

        res[idx] = I

    return res



In [3]:
def test_integration_speed(methods,func,a,b,N, true_integral, name,log_int=False):
    """
    funcs: dict con nome -> funzione di integrazione
    x, y_true: dati
    true_integral: valore analitico atteso
    """
    if log_int==False:
        x_full = np.linspace(a, b, N)
    else:
        a=a+(a/100000000)
        x_full = np.logspace(np.log10(a),np.log10(b), N)
    y_full = func(x_full)

    results = []
    for label, f in methods.items():
        t0 = time.time()
        val = f(y_full, x_full)
        t1 = time.time()
        err = abs(val - true_integral) / true_integral
        results.append((label, val, err, t1 - t0,N))
    print(f"\n============== {name} ==============")
    print(f"{'Metodo':<20}{'Valore':<15}{'Errore rel.':<15}{'Tempo (s)':<15}{'Points':<15}")
    print("")
    for r in results:
        print(f"{r[0]:<20}{r[1]:<15.8g}{r[2]:<15.3e}{r[3]:<15.3e}{r[4]:<15}")


def test_integration_speed2(methods,func,a,b,N, true_integral, name,log_int=False):
    """
    funcs: dict con nome -> funzione di integrazione
    x, y_true: dati
    true_integral: valore analitico atteso
    """

    if log_int:
        # lavoro in scala logaritmica
        log_a, log_b = np.log10(a), np.log10(b)
        x_full = np.logspace(log_a,log_b, N)
    else:
        x_full = np.linspace(a, b, N)
    
    y_full = func(x_full) 

    results = []
    for label, f in methods.items():

        if label.startswith('simp'):
            
        
            if log_int: 
                log_range=log_b-log_a
                h_T = (log_range) / (N - 1)
                h_S = h_T ** 0.5
                N_S = int(np.round((log_range)/h_S)) + 1         
                x_simp = np.logspace(np.log10(a), np.log10(b), N_S)

            else:
                h_T = (b - a) / (N - 1)  
                h_S = h_T ** 0.5
                N_S = int(np.round((b-a)/h_S)) + 1
                x_simp = np.linspace(a, b, N_S)   

            y_simp = func(x_simp)

            t0 = time.time()
            val = f(y_simp, x_simp)
            t1 = time.time()
            err = abs(val - true_integral) / true_integral
            results.append((label, val, err, t1 - t0,N_S))
        
        else:    
            t0 = time.time()
            val = f(y_full, x_full)
            t1 = time.time()
            err = abs(val - true_integral) / true_integral
            results.append((label, val, err, t1 - t0,N))
        
    print(f"\n============== {name} ==============")
    print(f"{'Metodo':<20}{'Valore':<15}{'Errore rel.':<15}{'Tempo (s)':<15}{'Points':<15}")
    print("")
    for r in results:
        print(f"{r[0]:<20}{r[1]:<15.8g}{r[2]:<15.3e}{r[3]:<15.3e}{r[4]:<15}")



In [6]:
import numpy as np
import time
from scipy.integrate import simps

# ---- Importa qui la tua funzione trapz_loglog ----
# def trapz_loglog(y, x, axis=-1, intervals=False): ...
# (usa la versione già definita nel tuo codice)

# ---- Funzione di test ----


N=100000
a=1.0
b=10.0
x=np.linspace(a,b,N)

funcs = {
    #"numpy_trapz": np.trapz,
    "trapz numba": trapz_numba,
    "trapz_loglog_nd_fast": trapz_loglog_nd_fast,
    #"simpson": simps,
    "simpson nu uniform":simps_numba_uniform,
}

# ---- Caso di test 1: -----------
y=lambda x: x**4  # funzione: y = x^2
true_int = (1/5) * (x[-1]**5 - x[0]**5)

test_integration_speed(funcs,y, a,b,N, true_int, "Power law y = x^3")


# ---- Caso di test 2: -------------------
x1=np.linspace(a,b,N*10)
sin = lambda x:np.sin(x) + 1.5  # positivo
true_int2 = np.trapz(sin(x1), x1)  # non abbiamo soluzione analitica
test_integration_speed(funcs, sin, a,b,N, true_int2, "Oscillante sin(x)")



============== Power law y = x^3 ==============
Metodo              Valore         Errore rel.    Tempo (s)      Points         

trapz numba         19999.8        1.349e-10      1.450e-04      100000         
trapz_loglog_nd_fast19999.8        1.783e-14      2.096e-01      100000         
simpson nu uniform  19999.8        9.459e-15      8.583e-05      100000         

============== Oscillante sin(x) ==============
Metodo              Valore         Errore rel.    Tempo (s)      Points         

trapz numba         14.879374      6.195e-11      9.465e-05      100000         
trapz_loglog_nd_fast14.879374      2.354e-10      2.333e-03      100000         
simpson nu uniform  14.879374      6.163e-13      8.392e-05      100000         


In [5]:


funcs = {
   # "numpy_trapz": np.trapz,
    "trapz numba": trapz_numba,
   # "simpson": simps,
    "simpson nu uniform":simps_numba_uniform,
}

# ---- Caso di test 1: -----------
# ---- Caso di test 1: -----------
y=lambda x: x**4  # funzione: y = x^2
true_int = (1/5) * (x[-1]**5 - x[0]**5)

test_integration_speed2(funcs,y,a,b,N, true_int, "Power law y = x^3")


# ---- Caso di test 2: -------------------
x1=np.linspace(a,b,N*10)
sin = lambda x:np.sin(x) + 1.5  # positivo
true_int2 = np.trapz(sin(x1), x1)  # non abbiamo soluzione analitica
test_integration_speed2(funcs, sin, a,b,N, true_int2, "Oscillante sin(x)")



============== Power law y = x^3 ==============
Metodo              Valore         Errore rel.    Tempo (s)      Points         

trapz numba         19999.8        1.349e-10      1.180e-04      100000         
simpson nu uniform  19999.8        4.869e-13      6.676e-06      950            

============== Oscillante sin(x) ==============
Metodo              Valore         Errore rel.    Tempo (s)      Points         

trapz numba         14.879374      6.195e-11      1.197e-04      100000         
simpson nu uniform  14.879374      4.734e-12      5.960e-06      950            


#### Log Log

In [ ]:

x1 = np.logspace(np.log10(a), np.log10(b), N)

funcs = {
   # "numpy_trapz": np.trapz,
    #"trapz_loglog": trapz_loglog,
    "trapz_loglog numba": trapz_loglog_nd_fast,
    "trapz numba": trapz_numba,
    #"simpson": simps,
    "simpson nu log2":simps_logspace_numba2,
}

# ---- Caso di test 1: -----------
# ---- Caso di test 1: -----------
y=lambda x: x**4  # funzione: y = x^2
true_int = (1/5) * (x1[-1]**5 - x1[0]**5)

test_integration_speed(funcs,y,a,b,N, true_int, "Power law y = x^3",log_int=True)


# ---- Caso di test 2: -------------------
sin = lambda x:np.sin(x) + 1.5  # positivo

true_int2 = np.trapz(sin(x1), x1)  # non abbiamo soluzione analitica
test_integration_speed(funcs, sin, a,b,N, true_int2, "Oscillante sin(x)",log_int=True)

In [ ]:

funcs = {
    #"numpy_trapz": np.trapz,
    #"trapz_loglog": trapz_loglog,
    "trapz_loglog N": trapz_loglog_nd,
    "trapz_loglog numba": trapz_loglog_nd_fast,
    "trapz numba": trapz_numba,
    #"simpson": simps,
    "simpson nu log2":simps_logspace_numba2,
}

# ---- Caso di test 1: -----------

test_integration_speed2(funcs,y,a,b,N, true_int, "Power law y = x^3",log_int=True)

test_integration_speed2(funcs, sin, a,b,N, true_int2, "Oscillante sin(x)",log_int=True)

In [ ]:
def integrate_auto(y, x, method="trapz", log_int=False):
    """
    Integra y(x) scegliendo tra integrazione lineare o logaritmica.
    Se method='simpson', riduce i punti per migliorare la velocità.
    
    Parameters
    ----------
    y : array
        Valori della funzione.
    x : array
        Ascisse (lineari o logaritmiche).
    method : str
        'trapz' o 'simpson'
    log_int : bool
        True se x è logaritmico (es. np.logspace), False se lineare.
    """

    a, b = x[0], x[-1]
    N = len(x)
    if method == "trapz":
        if log_int:
            # integrazione log-log
            B=trapz_loglog_nd_fast(y ,x)
            return B
        else:
            B=trapz_numba(y, x)
            return B

    elif method == "simps":
        if log_int:
            # riscalo punti in scala log
            log_a, log_b = np.log10(a), np.log10(b)
            log_range = log_b - log_a
            h_T = log_range / (N - 1)
            h_S = h_T ** 0.5
            N_S = int(np.round(log_range / h_S)) + 1
            #print(N_S)
            x_s = np.logspace(log_a, log_b, N_S)
            y_s = np.interp(x_s, x, y)
            A=simps_logspace_numba2(y_s, x_s)
            return  A
        else:
            h_T = (b - a) / (N - 1)
            h_S = h_T ** 0.5
            N_S = int(np.round((b - a) / h_S)) + 1
            #print(N_S)
            x_s = np.linspace(a, b, N_S)
            y_s = np.interp(x_s, x, y)
            A=simps_numba_uniform(y_s,x_s)
            return A
    else:
        raise ValueError(f"Metodo '{method}' non riconosciuto.")

In [ ]:
def simps_points(a,b,N,loglog=False):
    
    if loglog:
        log_a, log_b = np.log10(a), np.log10(b)
        log_range = log_b - log_a
        h_T = log_range / (N - 1)
        h_S = h_T ** 0.5
        N_S = int(np.round(log_range / h_S)) + 1
        x_s = np.logspace(log_a, log_b, N_S)

    else:
        h_T = (b - a) / (N - 1)
        h_S = h_T ** 0.5
        N_S = int(np.round((b - a) / h_S)) + 1
        x_s = np.linspace(a, b, N_S)
    return x_s

In [ ]:
A=simps_points(1,1000,200,loglog="True")
print(A)
print(type(A))

In [ ]:

from tqdm import tqdm
# Parametri
N = 10000000
a, b = 1.0, 10.0
x = np.linspace(a, b, N)
y = x**2   # funzione di test
Z = 1500    # numero di ripetizioni

# Liste per i tempi
times_trapz = []
integrale_trapz=[]
times_simpson = []
integrale_simpson=[]

for _ in tqdm(range(Z),desc="Benchmarking"):
    # Trapz
    start = time.time()
    A=integrate_auto(y, x, method="trapz", log_int=False)
    end = time.time()
    times_trapz.append(end - start)
    integrale_trapz.append(A)

    # Simpson
    start = time.time()
    B=integrate_auto(y, x, method="simps", log_int=False)
    end = time.time()
    times_simpson.append(end - start)
    integrale_simpson.append(B)

# Scatter plot dei tempi
plt.figure(figsize=(10,6))
plt.scatter(range(Z), times_trapz, color='blue', label='Trapz', alpha=0.7)
plt.scatter(range(Z), times_simpson, color='red', label='Simpson', alpha=0.7)
plt.xlabel("Iterazione")
plt.ylabel("Tempo di esecuzione [s]")
plt.yscale('log')
plt.title("Confronto tempi di esecuzione tra Trapz e Simpson")
plt.legend()
plt.grid(True)
plt.show()

# Stampe statistiche
print(f"Trapz: media = {np.mean(times_trapz):.3e} s, std = {np.std(times_trapz):.3e} s")
print(f"Simpson: media = {np.mean(times_simpson):.3e} s, std = {np.std(times_simpson):.3e} s")


In [ ]:

from tqdm import tqdm
# Parametri
N = 1000
a, b = 1.0, 10.0
x = np.linspace(a, b, N)
y = x**4   # funzione di test
y=sin(x)
Z = 1500    # numero di ripetizioni

# Liste per i tempi
times_trapz = []
integrale_trapz=[]
times_simpson = []
integrale_simpson=[]

for _ in tqdm(range(Z),desc="Benchmarking"):
    # Trapz
    start = time.time()
    A=integrate_auto(y, x, method="trapz", log_int=True)
    end = time.time()
    times_trapz.append(end - start)
    integrale_trapz.append(A)

    # Simpson
    start = time.time()
    B=integrate_auto(y, x, method="simps", log_int=True)
    end = time.time()
    times_simpson.append(end - start)
    integrale_simpson.append(B)

# Scatter plot dei tempi
plt.figure(figsize=(10,6))
plt.scatter(range(Z), times_trapz, color='blue', label='Trapz', alpha=0.7)
plt.scatter(range(Z), times_simpson, color='red', label='Simpson', alpha=0.7)
plt.xlabel("Iterazione")
plt.ylabel("Tempo di esecuzione [s]")
plt.yscale('log')
plt.title("Confronto tempi di esecuzione tra Trapz e Simpson")
plt.legend()
plt.grid(True)
plt.show()

# Stampe statistiche
print(f"Trapz: media = {np.mean(times_trapz):.3e} s, std = {np.std(times_trapz):.3e} s")
print(f"Simpson: media = {np.mean(times_simpson):.3e} s, std = {np.std(times_simpson):.3e} s")


In [ ]:
N = 1000000
a, b = 1.0, 10.0
x = np.linspace(a, b, N)
y = x**2   # funzione di test
Z = 1500    # numero di ripetizioni

# Liste per i tempi
times_trapz = []
times_simpson = []

for _ in tqdm(range(Z),desc="Benchmarking"):
    # Trapz
    start = time.time()
    #integrate_auto(y, x, method="trapz", log_int=False)
    trapz_numba(y,x)
    end = time.time()
    times_trapz.append(end - start)

    # Simpson
    start = time.time()
    #ntegrate_auto(y, x, method="simps", log_int=False)
    simps_numba_uniform(y,x)
    end = time.time()
    times_simpson.append(end - start)

# Scatter plot dei tempi
plt.figure(figsize=(10,6))
plt.scatter(range(Z), times_trapz, color='blue', label='Trapz', alpha=0.7)
plt.scatter(range(Z), times_simpson, color='red', label='Simpson', alpha=0.7)
plt.yscale('log')
plt.xlabel("Iterazione")
plt.ylabel("Tempo di esecuzione [s]")
plt.title("Confronto tempi di esecuzione tra Trapz e Simpson")
plt.legend()
plt.grid(True)
plt.show()

# Stampe statistiche
print(f"Trapz: media = {np.mean(times_trapz):.3e} s, std = {np.std(times_trapz):.3e} s")
print(f"Simpson: media = {np.mean(times_simpson):.3e} s, std = {np.std(times_simpson):.3e} s")

In [ ]:
def gamma_array(Eemin, Eemax, nEed):
    """
    Genera un array di fattori di Lorentz distribuiti in scala logaritmica.
    """
    log10gmin = np.log10(Eemin)
    log10gmax = np.log10(Eemax)
    return np.logspace(log10gmin, log10gmax, max(10, int(nEed * (log10gmax - log10gmin))))

# ------------------------------------------------------------
# 2️⃣ Esempio di distribuzione di particelle
# ------------------------------------------------------------
def particle_distribution(E):
    """
    Esempio: distribuzione di energia a legge di potenza.
    Qui puoi cambiare il modello a piacere.
    """
    p = 5  # indice spettrale
    K = 1e10  # normalizzazione arbitraria
    return K * E**(-p)

# ------------------------------------------------------------
# 3️⃣ Densità di particelle per unità di gamma
# ------------------------------------------------------------
def nelec_array(gamma):
    """
    Restituisce n(gamma), cioè particelle per unità di fattore di Lorentz.
    """
    pd = particle_distribution(gamma)
    return pd 
def Etot(y,x):
    """Total energy in electrons used for the radiative calculation"""
    Etot = trapz_loglog(y, x)
    return Etot

def Etot_simp(y,x):
    """Total energy in electrons used for the radiative calculation"""
    Etot_simp=simps_logspace_numba2(y,x)
    return Etot_simp

In [ ]:
emin=1e4
emax=1e8
nEed_values = np.logspace(1, 5, 20, dtype=int)# da 10 a 10^4 punti
E_values = []

for n in nEed_values:
    gamma = gamma_array(emin, emax, nEed=n)
    print(len(gamma))
    nelec = nelec_array(gamma)
    E = Etot(gamma * nelec, gamma)
    E_values.append(E)
    
    sim_array=simps_points(emin,emax,n)
    Esim = Etot_simp(gamma * nelec, gamma)


# ==== Plot ====

plt.figure(figsize=(10,6))
plt.plot(nEed_values, E_values, 'o-', label='E_tot(nEed)')
plt.xscale('log')
plt.xlabel("Numero di punti nEed")
plt.ylabel("E_tot (integrale)")
plt.title("Convergenza di E_tot al variare di nEed")
plt.grid(True, which='both', ls='--', alpha=0.6)
plt.legend()
plt.show()


In [ ]:

x1 = np.logspace(np.log10(a), np.log10(b), N)

funcs = {
   # "numpy_trapz": np.trapz,
    #"trapz_loglog": trapz_loglog,
    "trapz_loglog numba": trapz_loglog_nd_fast,
    "trapz numba": trapz_numba,
    #"simpson": simps,
    "simpson nu log2":simps_logspace_numba2,
}

# ---- Caso di test 1: -----------
# ---- Caso di test 1: -----------
y=lambda x: x**4  # funzione: y = x^2
true_int = (1/5) * (x1[-1]**5 - x1[0]**5)

test_integration_speed(funcs,y,a,b,N, true_int, "Power law y = x^3",log_int=True)


# ---- Caso di test 2: -------------------
sin = lambda x:np.sin(x) + 1.5  # positivo

true_int2 = np.trapz(sin(x1), x1)  # non abbiamo soluzione analitica
test_integration_speed(funcs, sin, a,b,N, true_int2, "Oscillante sin(x)",log_int=True)

In [ ]:
import pandas as pd
import math



# --- Esperimento di confronto ---
def run_experiment(f, a, b, true_value=None, Ns=None, label="f"):
    """
    Confronta errore della regola del trapezio e di Simpson su diversi N.
    """
    if Ns is None:
        Ns = [21, 41, 81, 161, 321, 641, 1281,2000,4000,10000]  # N dispari
    trap_errors, simp_errors,trap_numba_errors = [], [],[]
    hs, trap_vals, simp_vals,trap_numba_vals= [], [], [],[]
    
    # Valore di riferimento ad alta risoluzione (se non fornito)
    if true_value is None:
        Nref = 200_000
        xref = np.linspace(a, b, Nref)
        true_value = np.trapz(f(xref), xref)
    
    for N in Ns:

        x = np.linspace(a, b, N)
        #x = np.logspace(np.log10(a), np.log10(b), N)
        I_trap = np.trapz(f(x), x)
        I_trap_numba = trapz_loglog_nd_fast(f(x), x)
        I_simp = simps_logspace_numba2(f(x), x)
       
        trap_vals.append(I_trap)
        trap_numba_vals.append(I_trap_numba)
        simp_vals.append(I_simp)

        trap_errors.append(abs(I_trap - true_value))
        trap_numba_errors.append(abs(I_trap_numba- true_value))
        simp_errors.append(abs(I_simp - true_value))
        hs.append((b - a) / (N - 1))
    
    # Fit log-log per stimare l'ordine di convergenza
    logh = np.log(hs)
    trap_slope = np.polyfit(logh, np.log(trap_errors), 1)[0]
    trap_numba_slope = np.polyfit(logh, np.log(trap_numba_errors), 1)[0]
    simp_slope = np.polyfit(logh, np.log(simp_errors + np.finfo(float).eps), 1)[0]
    
    # Grafico errori
    plt.figure(figsize=(10, 6))
    plt.loglog(hs, trap_errors, marker='o', label=f"Trapezio (ordine ~ {-trap_slope:.2f})")
    plt.loglog(hs, trap_numba_errors, marker='o', label=f"Trapezio numba (ordine ~ {-trap_numba_slope:.2f})")
    plt.loglog(hs, simp_errors, marker='s', label=f"Simpson (ordine ~ {-simp_slope:.2f})")
    plt.xlabel("Passo h")
    plt.ylabel("Errore assoluto")
    plt.title(f"Confronto Trapezio vs Simpson per {label}")
    plt.legend()
    plt.grid(True, which="both", ls=":")
    plt.show()

    # Tabella risultati
    df = pd.DataFrame({
        "N": Ns,
        "h": hs,
        "I_trap": trap_vals,
        "I_simp": simp_vals,
        "err_trap": trap_errors,
        "err_simp": simp_errors
    })
    print(f"\nOrdini stimati per {label}: trap ~ {-trap_slope:.2f}, trap numba ~ {-trap_numba_slope:.2f}, simpson ~ {-simp_slope:.2f}")
    #print(df)
    return df

# --- Test su tre funzioni ---


# 2. f(x) = sin(x) su [0, pi], integrale esatto = 2
f2 = lambda x: np.sin(x)

display(run_experiment(f2, 0.0, math.pi, true_value=2.0, label="sin(x) [0,pi]"))

# 3. f(x) = exp(-x^2) su [0,1]
f3 = lambda x: np.exp(-x**2)
# Calcoliamo riferimento numerico
Nref = 400_000
xref = np.linspace(0.0, 1, Nref)
true3 = np.trapz(f3(xref), xref)

display(run_experiment(f3, 0.0, 1.0, true_value=true3, label="exp(-x^2) [0,1]"))


In [ ]:
def test_integration_speed(funcs,y,a,b,N, true_integral, name):
    """
    funcs: dict con nome -> funzione di integrazione
    x, y_true: dati
    true_integral: valore analitico atteso
    """
    x_full = np.linspace(a, b, N)
    y_full = y(x_full)

    results = []
    for label, f in funcs.items():

        if label.startswith('simp'):
            # passo h_T
            h_T = (b-a)/(N-1)

            # usando formula teorica: h_S ~ h_T^(1/2)
            h_S = h_T**0.5
            N_S = int(np.round((b-a)/h_S)) + 1
                
            x_simp = np.linspace(a, b, N_S)
            y_simp = y(x_simp)

            t0 = time.time()
            val = f(y_simp, x_simp)
            t1 = time.time()
            err = abs(val - true_integral) / true_integral
            results.append((label, val, err, t1 - t0))
        
        else:

            t0 = time.time()
            val = f(y_full, x_full)
            t1 = time.time()
            err = abs(val - true_integral) / true_integral
            results.append((label, val, err, t1 - t0))
        
    print(f"\n======== {name} ========")
    print(f"{'Metodo':<20}{'Valore':<15}{'Errore rel.':<15}{'Tempo (s)':<15}")
    print("")
    for r in results:
        print(f"{r[0]:<20}{r[1]:<15.6g}{r[2]:<15.3e}{r[3]:<15.3e}")

N = 10000
a=0.0
b=10.0

funcs = {
    "numpy_trapz": np.trapz,
    "trapz numba": trapz_numba,
    "simpson": simps,
    "simpson nu uniform":simps_numba_uniform,
}

# ---- Caso di test 1: -----------

y = lambda x: x**3
true_int = (1/4) * (x[-1]**4 - x[0]**4)

test_integration_speed(funcs, y,a,b,N, true_int, "Power law y = x^3")

# ---- Caso di test 2: -------------------
y2 = np.sin(x / 10) + 1.5  # positivo
true_int2 = np.trapz(y2, x)  # non abbiamo soluzione analitica
test_integration_speed(funcs,y,a,b,N, true_int2, "Oscillante sin(x/1000)")

In [ ]:
import numpy as np
from scipy.integrate import simps

def integrate_optimized(f, a, b, N, method='trapz', use_numba=False):
    """
    Integra f(x) su [a,b] con N punti uniformi.
    Se method='simpson', riduce il numero di punti per avere approssimazione simile a trapz.
    """
    x_full = np.linspace(a, b, N)
    y_full = f(x_full)

    # 1. Trapezio completo come riferimento
    if use_numba:
        # assumendo trapz_numba esista
        I_trap = trapz_numba(y_full, x_full)
    else:
        I_trap = np.trapz(y_full, x_full)

    # Se vogliamo Simpson ottimizzato
    if method.startswith('simp'):
        # passo h_T
        h_T = (b-a)/(N-1)
        # Numero punti Simpson stimato per stesso errore
        # usando formula teorica: h_S ~ h_T^(1/2)
        h_S = h_T**0.5
        N_S = int(np.round((b-a)/h_S)) + 1
        # rendiamo N_S dispari per Simpson
        if N_S % 2 == 0:
            N_S += 1

        # Creiamo sottoinsieme dei punti
        x_simp = np.linspace(a, b, N_S)
        y_simp = f(x_simp)

        if use_numba:
            I_simp = simps_numba_uniform(y_simp, x_simp)
        else:
            I_simp = simps(y_simp, x_simp)
        return I_trap, I_simp, N_S

    return I_trap, N


In [ ]:
f = lambda x: np.sin(x)
f3 = lambda x: np.exp(-x**4)
a, b = 0, np.pi
N = 10000000

# Trapezio vs Simpson ottimizzato
I_trap, I_simp, N_simp = integrate_optimized(f3, a, b, N, method='simpson')
print(f"Trapezio su {N} punti = {I_trap:.8f}")
print(f"Simpson ottimizzato su {N_simp} punti = {I_simp:.8f}")
print(f"Riduzione punti: {N/N_simp:.1f}x")


In [ ]:

import numpy as np
import time
from scipy.integrate import simps, quad, dblquad

# ---------------------------
# Fattore Doppler semplificato
# ---------------------------
def doppler_factor(Gamma, theta, theta_obs, phi):
    beta = np.sqrt(1 - 1/Gamma**2)
    cos_psi = np.cos(theta)*np.cos(theta_obs) + np.sin(theta)*np.sin(theta_obs)*np.cos(phi)
    return 1.0/(Gamma*(1 - beta*cos_psi))

# ---------------------------
# Parametri
# ---------------------------
Gamma = 50.0
theta_obs = np.deg2rad(10.0)   # osservatore
Ntheta = 20                   # punti in theta
Nphi   = 40                   # punti in phi

# Griglie
theta_vals = np.linspace(0, np.pi/2, Ntheta)   # mezzo getto (0–90°)
phi_vals   = np.linspace(0, 2*np.pi, Nphi)

Theta, Phi = np.meshgrid(theta_vals, phi_vals, indexing='ij')

# Integrando: δ(θ,φ) * sinθ
integrand = doppler_factor(Gamma, Theta, theta_obs, Phi) * np.sin(Theta)

# ---------------------------
# Integrazione con trapz
# ---------------------------
t0 = time.time()
int_phi = np.trapz(integrand, phi_vals, axis=1)   # integra su phi
res_trapz = np.trapz(int_phi, theta_vals)         # integra su theta
t1 = time.time()
print(f"Trapz: {res_trapz:.6f}, tempo = {t1-t0:.6e} s")

# ---------------------------
# Integrazione con simps
# ---------------------------
t0 = time.time()
int_phi_s = simps(integrand, phi_vals, axis=1)
res_simps = simps(int_phi_s, theta_vals)
t1 = time.time()
print(f"Simps: {res_simps:.6f}, tempo = {t1-t0:.6e} s")

# ---------------------------
# Integrazione con dblquad
# ---------------------------
t0 = time.time()
res_quad, err_quad = dblquad(
    lambda phi, theta: doppler_factor(Gamma, theta, theta_obs, phi) * np.sin(theta),
    0, np.pi/2,    # limiti in theta
    lambda theta: 0, lambda theta: 2*np.pi   # limiti in phi
)
t1 = time.time()
print(f"DblQuad: {res_quad:.6f}, tempo = {t1-t0:.6e} s (errore stimato {err_quad:.2e})")


In [ ]:
# =============================
# 1. Definizione delle funzioni
# =============================

def doppler_factor(Gamma, theta, theta_obs, phi):
    beta = np.sqrt(1 - 1/Gamma**2)
    cos_psi = np.cos(theta)*np.cos(theta_obs) + np.sin(theta)*np.sin(theta_obs)*np.cos(phi)
    return 1.0 / (Gamma * (1 - beta * cos_psi))


# =============================
# 2. Setup parametri e griglie
# =============================

Gamma = 50.0
theta_obs = np.deg2rad(10.0)
Ntheta = 10000
Nphi = 50000

theta_vals = np.linspace(0, np.pi/2, Ntheta)
phi_vals = np.linspace(0, 2*np.pi, Nphi)

                    
#theta_vals = np.logspace(np.log10(1e-3), np.log10(np.pi/2), Ntheta)
#phi_vals = np.logspace(np.log10(1e-3), np.log10(2*np.pi), Ntheta)

Theta, Phi = np.meshgrid(theta_vals, phi_vals, indexing='ij')
integrand = (doppler_factor(Gamma, Theta, theta_obs, Phi)**2) * np.sin(Theta)


start = time.time()
I_phi_tot = np.trapz(integrand, phi_vals, axis=1)
I_theta_tot= np.trapz(I_phi_tot, theta_vals)
time_tot= time.time() - start

In [ ]:
print(f"Integrale totale (trapz)             :  {I_theta_tot:.12e}   in {time_tot :.3f}s")

In [ ]:

Gamma = 50.0
theta_obs = np.deg2rad(10.0)
Ntheta = 1000
Nphi = 5000

theta_vals = np.linspace(0, np.pi/2, Ntheta)
phi_vals = np.linspace(0, 2*np.pi, Nphi)

                    
#theta_vals = np.logspace(np.log10(1e-3), np.log10(np.pi/2), Ntheta)
#phi_vals = np.logspace(np.log10(1e-3), np.log10(2*np.pi), Ntheta)

Theta, Phi = np.meshgrid(theta_vals, phi_vals, indexing='ij')
integrand = (doppler_factor(Gamma, Theta, theta_obs, Phi)**2) * np.sin(Theta)


# =============================
# 3. Integrazione su φ
# =============================

start = time.time()
I_phi_trapz = np.trapz(integrand, phi_vals, axis=1)
t_phi_trapz = time.time() - start

start = time.time()
I_phi_trapz_numba= trapz_numba(integrand, phi_vals)
t_phi_trapz_numba = time.time() - start

start = time.time()
I_phi_simps = simps(integrand, phi_vals, axis=1)
t_phi_simps = time.time() - start

start = time.time()
I_phi_simps_numba = simps_numba(integrand, phi_vals,step=1)
t_phi_simps_numba = time.time() - start


start = time.time()
I_phi_loglog = trapz_loglog(integrand, phi_vals)
t_phi_loglog = time.time() - start

start = time.time()
I_phi_loglog_numba = trapz_loglog_nd(integrand, phi_vals)
t_phi_loglog_numba = time.time() - start

start = time.time()
I_phi_loglog_numba_fast = trapz_loglog_nd_fast(integrand, phi_vals)
t_phi_loglog_numba_fast= time.time() - start



# =============================
# 4. Ora integri anche su θ
# =============================
start = time.time()
I_theta_trapz = np.trapz(I_phi_trapz, theta_vals)
t_theta_trapz = time.time() - start

start = time.time()
I_theta_trapz_numba=trapz_numba(I_phi_trapz, theta_vals)
t_theta_trapz_numba= time.time() - start

start = time.time()
I_theta_simps = simps(I_phi_simps, theta_vals,dx=4)
t_theta_simps = time.time() - start

start = time.time()
I_theta_simps_numba = simps_numba(I_phi_simps, theta_vals,step=1)
t_theta_simps_numba= time.time() - start


start = time.time()
I_theta_loglog = trapz_loglog(I_phi_loglog, theta_vals)
t_theta_loglog = time.time() - start

start = time.time()
I_theta_loglog_numba= trapz_loglog_nd(I_phi_loglog_numba, theta_vals)
t_theta_loglog_numba = time.time() - start

start = time.time()
I_theta_loglog_numba_fast= trapz_loglog_nd(I_phi_loglog_numba_fast, theta_vals)
t_theta_loglog_numba_fast = time.time() - start



# =============================
# 5. Risultati
# =============================
print("\n=== RISULTATI DOPPLER INTEGRAZIONE ===")
print(f"Integrale totale (trapz)             :  {I_theta_trapz:.6e}   in {t_theta_trapz+t_phi_trapz:.3f}s")
print(f"Integrale totale (trapz numba)       :  {I_theta_trapz_numba:.6e}   in {t_theta_trapz_numba+t_phi_trapz_numba:.3f}s")
print(f"Integrale totale (simpson)           :  {I_theta_simps:.6e}   in {t_theta_simps+t_phi_simps:.3f}s")
print(f"Integrale totale (simpson numba)     :  {I_theta_simps_numba:.6e}   in {t_theta_simps_numba+t_phi_simps_numba:.3f}s")
print(f"Integrale totale (loglog)            :  {I_theta_loglog:.6e}   in {t_theta_loglog+t_phi_loglog:.3f}s")
print(f"Integrale totale (loglog numba)      :  {I_theta_loglog_numba:.6e}   in {t_theta_loglog_numba+t_phi_loglog_numba:.3f}s")
print(f"Integrale totale (loglog numba fast) :  {I_theta_loglog_numba_fast:.6e}   in {t_theta_loglog_numba_fast+t_phi_loglog_numba_fast:.3f}s")

print("\nDifferenze relative:")
print(f"simpson vs trapz:  {(I_theta_simps_numba - I_theta_trapz_numba)/I_theta_trapz_numba:.3e}")
print(f"loglog  vs trapz:  {(I_theta_loglog_numba_fast- I_theta_trapz_numba)/I_theta_trapz_numba:.3e}")


In [ ]:

from scipy.integrate import quad, simps



def integrand(theta, phi):
    """Integrando completo con sin(theta) per misura sferica"""
    return (doppler_factor(Gamma, theta, theta_obs, phi)**2) * np.sin(theta)

def integrale_2D():
    # integrazione esterna su θ
    def integrand_phi_integrated(theta):
        # integrazione interna su φ 
        res_phi, err_phi = quad(lambda phi: integrand(theta, phi), 0.0, 2*np.pi,
                                epsabs=1e-10, epsrel=1e-10)
        return res_phi
    res_theta, err_theta = quad(integrand_phi_integrated, 0.0, np.pi/2,
                                epsabs=1e-10, epsrel=1e-10)
    return res_theta, err_theta


# --- Riferimento: integrazione 1D in theta con quad (alta accuratezza) ---
start = time.time()
I_ref, err_ref = integrale_2D()
print(time.time()-start)
print("Riferimento (quad su theta dopo integrazione analitica su phi):")
print("I_ref = {:.12e}, stima errore = {:.2e}".format(I_ref, err_ref))


# --- 2) integrazione 2D numerica su griglia (trapz e simpson)
def integrate_2d_trapz_simpson(Gamma, theta_obs, Ntheta, Nphi):
    
    theta = np.linspace(0.0, np.pi/2, Ntheta)
    phi = np.linspace(0.0, 2.0*np.pi, Nphi)
    
    Theta, Phi = np.meshgrid(theta, phi, indexing='ij')  # Theta.shape = (Ntheta, Nphi)
    D = doppler_factor(Gamma, Theta, theta_obs, Phi)**2
    integrand = D * np.sin(Theta)


    # trapz: integra prima su phi (axis=1), poi su theta (axis=0)
    print("")
    start=time.time()
    I_trap_phi = np.trapz(integrand, phi, axis=1)
    I_trap = np.trapz(I_trap_phi, theta, axis=0)
    time_trap=time.time()-start

    start=time.time()
    I_trap_phi = trapz_numba(integrand, phi)
    I_trap_numba = trapz_numba(I_trap_phi, theta)
    time_trap_numba=time.time()-start

    start=time.time()
    I_simp_phi = simps(integrand, phi, axis=1)
    I_simp = simps(I_simp_phi, theta, axis=0)
    time_simps=time.time()-start
    
    start=time.time()
    I_simp_phi_numba = simps_numba(integrand, phi)
    I_simp_numba = simps_numba(I_simp_phi_numba, theta)
    time_simp_numba=time.time()-start

    start=time.time()
    I_loglog_phi = trapz_loglog_nd_fast(integrand, phi)
    I_loglog = trapz_loglog_nd_fast(I_loglog_phi, theta)
    time_trap_loglog=time.time()-start

    ######################################################

    print(f"Integrale (trapz):        {I_trap:.6e}   in {time_trap:.4f}s")
    print(f"Integrale (trapz numba):  {I_trap_numba:.6e}   in {time_trap_numba:.4f}s")
    print(f"Integrale (simps):        {I_simp:.6e}   in {time_simps:.4f}s")
    print(f"Integrale (simps numba):  {I_simp_numba:.6e}   in {time_simp_numba:.4f}s")
    print(f"Integrale (loglog numba):  {I_loglog:.6e}   in {time_trap_loglog:.4f}s")

    return (I_trap,I_trap_numba, I_simp,I_simp_numba,I_loglog,
            time_trap, time_trap_numba, time_simps, time_simp_numba, time_trap_loglog)

# Test di convergenza: varie risoluzioni
resolutions = [
    (400,400),
    (800, 800),
    (1200, 1200),
    (1600, 1600),
    (2000, 2000),
    (2400, 2400),
    (6000, 6000),
    (10000,10000),
]

results = []
for Ntheta, Nphi in resolutions:
    print("")
    print("===============================================================")
    print(f"Ntheta={Ntheta}, Nphi={Nphi}")
    (I_trap, I_trap_numba, I_simp, I_simp_numba, I_loglog,
     time_trap, time_trap_numba, time_simps, time_simp_numba, time_trap_loglog) = \
        integrate_2d_trapz_simpson(Gamma, theta_obs, Ntheta, Nphi)
    
    err_trap = abs(I_trap - I_ref)
    err_trap_numba = abs(I_trap_numba - I_ref)
    err_simp = abs(I_simp - I_ref)
    err_simp_numba = abs(I_simp_numba - I_ref)
    err_loglog= abs(I_loglog - I_ref)
    
        # Poi li salvi qui:
    results.append({
        "Ntheta": Ntheta, "Nphi": Nphi,
        "err_trap": err_trap, "err_trap_numba": err_trap_numba,
        "err_simp": err_simp, "err_simp_numba": err_simp_numba,
        "err_loglog": err_loglog,
        "t_trap": time_trap, "t_trap_numba": time_trap_numba,
        "t_simp": time_simps, "t_simp_numba": time_simp_numba,
        "t_loglog": time_trap_loglog
    })
    
    print("")
    print(f"Trapz=         {I_trap:.12e}, err={err_trap:.2e}")
    print(f"Trapz numba =  {I_trap_numba:.12e}, err={err_trap_numba:.2e}")
    print(f"Simpson=       {I_simp:.12e}, err={err_simp:.2e}")
    print(f"Simpson numba= {I_simp_numba:.12e}, err={err_simp_numba:.2e}")
    print(f"trap loglog=   {I_loglog:.12e}, err={err_loglog:.2e}")
    print("===============================================================")



plt.figure(figsize=(10,6))
Ns = [r["Ntheta"] for r in results]

plt.loglog(Ns, [r["err_trap"] for r in results], 'o-', label='Trapz')
plt.loglog(Ns, [r["err_trap_numba"] for r in results], 's--', label='Trapz (Numba)')
plt.loglog(Ns, [r["err_simp"] for r in results], 'o-', label='Simpson')
plt.loglog(Ns, [r["err_simp_numba"] for r in results], 's--', label='Simpson (Numba)')
plt.loglog(Ns, [r["err_loglog"] for r in results], 'd-.', label='Trapz Log-Log')

plt.xlabel("Nθ (risoluzione angolare)")
plt.ylabel("Errore assoluto |I - I_ref|")
plt.title("Convergenza dei metodi 2D vs riferimento")
plt.grid(True, which='both', ls='--', lw=0.6)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10,6))
plt.loglog(Ns, [r["t_trap"] for r in results], 'o-', label='Trapz')
plt.loglog(Ns, [r["t_trap_numba"] for r in results], 's--', label='Trapz (Numba)')
plt.loglog(Ns, [r["t_simp"] for r in results], 'o-', label='Simpson')
plt.loglog(Ns, [r["t_simp_numba"] for r in results], 's--', label='Simpson (Numba)')
plt.loglog(Ns, [r["t_loglog"] for r in results], 'd-.', label='Trapz Log-Log')

plt.xlabel("Nθ (risoluzione)")
plt.ylabel("Tempo di esecuzione [s]")
plt.title("Scalabilità in tempo dei metodi di integrazione 2D")
plt.grid(True, which='both', ls='--', lw=0.6)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:



# --- 2) integrazione 2D numerica su griglia (trapz e simpson)
def integrate_2d_trapz_simpson(Gamma, theta_obs, Ntheta, Nphi):
    
    theta_min = 1e-6      # rad
    theta_max = np.pi/2   # rad
    phi_min = 1e-6  # rad
    phi_max = 2*np.pi  # rad
    theta = np.concatenate(([0.0], np.logspace(np.log10(theta_min), np.log10(theta_max), Ntheta-1)))
    phi = np.concatenate(([0.0], np.logspace(np.log10(phi_min), np.log10(phi_max), Nphi-1)))
    
    Theta, Phi = np.meshgrid(theta, phi, indexing='ij')  # Theta.shape = (Ntheta, Nphi)
    D = doppler_factor(Gamma, Theta, theta_obs, Phi)**2
    integrand = D * np.sin(Theta)


    # trapz: integra prima su phi (axis=1), poi su theta (axis=0)
    print("")
    start=time.time()
    I_trap_phi = np.trapz(integrand, phi, axis=1)
    I_trap = np.trapz(I_trap_phi, theta, axis=0)
    time_trap=time.time()-start

    start=time.time()
    I_trap_phi = trapz_numba(integrand, phi)
    I_trap_numba = trapz_numba(I_trap_phi, theta)
    time_trap_numba=time.time()-start

    start=time.time()
    I_simp_phi = simps(integrand, phi, axis=1)
    I_simp = simps(I_simp_phi, theta, axis=0)
    time_simps=time.time()-start
    
    start=time.time()
    I_simp_phi_numba = simps_numba(integrand, phi)
    I_simp_numba = simps_numba(I_simp_phi_numba, theta)
    time_simp_numba=time.time()-start

    start=time.time()
    I_loglog_phi = trapz_loglog_nd_fast(integrand, phi)
    I_loglog = trapz_loglog_nd_fast(I_loglog_phi, theta)
    time_trap_loglog=time.time()-start

    ######################################################

    print(f"Integrale (trapz):        {I_trap:.6e}   in {time_trap:.4f}s")
    print(f"Integrale (trapz numba):  {I_trap_numba:.6e}   in {time_trap_numba:.4f}s")
    print(f"Integrale (simps):        {I_simp:.6e}   in {time_simps:.4f}s")
    print(f"Integrale (simps numba):  {I_simp_numba:.6e}   in {time_simp_numba:.4f}s")
    print(f"Integrale (loglog numba):  {I_loglog:.6e}   in {time_trap_loglog:.4f}s")

    return (I_trap,I_trap_numba, I_simp,I_simp_numba,I_loglog,
            time_trap, time_trap_numba, time_simps, time_simp_numba, time_trap_loglog)

# Test di convergenza: varie risoluzioni
resolutions = [
    (400,400),
    (800, 800),
    (1200, 1200),
    (1600, 1600),
    (2000, 2000),
    (2400, 2400),
    (6000, 6000),
    (10000,10000),
]

results = []
for Ntheta, Nphi in resolutions:
    print("")
    print("===============================================================")
    print(f"Ntheta={Ntheta}, Nphi={Nphi}")
    (I_trap, I_trap_numba, I_simp, I_simp_numba, I_loglog,
     time_trap, time_trap_numba, time_simps, time_simp_numba, time_trap_loglog) = \
        integrate_2d_trapz_simpson(Gamma, theta_obs, Ntheta, Nphi)
    
    err_trap = abs(I_trap - I_ref)
    err_trap_numba = abs(I_trap_numba - I_ref)
    err_simp = abs(I_simp - I_ref)
    err_simp_numba = abs(I_simp_numba - I_ref)
    err_loglog= abs(I_loglog - I_ref)
    
        # Poi li salvi qui:
    results.append({
        "Ntheta": Ntheta, "Nphi": Nphi,
        "err_trap": err_trap, "err_trap_numba": err_trap_numba,
        "err_simp": err_simp, "err_simp_numba": err_simp_numba,
        "err_loglog": err_loglog,
        "t_trap": time_trap, "t_trap_numba": time_trap_numba,
        "t_simp": time_simps, "t_simp_numba": time_simp_numba,
        "t_loglog": time_trap_loglog
    })
    
    print("")
    print(f"Trapz=         {I_trap:.12e}, err={err_trap:.2e}")
    print(f"Trapz numba =  {I_trap_numba:.12e}, err={err_trap_numba:.2e}")
    print(f"Simpson=       {I_simp:.12e}, err={err_simp:.2e}")
    print(f"Simpson numba= {I_simp_numba:.12e}, err={err_simp_numba:.2e}")
    print(f"trap loglog=   {I_loglog:.12e}, err={err_loglog:.2e}")
    print("===============================================================")



plt.figure(figsize=(10,6))
Ns = [r["Ntheta"] for r in results]

plt.loglog(Ns, [r["err_trap"] for r in results], 'o-', label='Trapz')
plt.loglog(Ns, [r["err_trap_numba"] for r in results], 's--', label='Trapz (Numba)')
plt.loglog(Ns, [r["err_simp"] for r in results], 'o-', label='Simpson')
plt.loglog(Ns, [r["err_simp_numba"] for r in results], 's--', label='Simpson (Numba)')
plt.loglog(Ns, [r["err_loglog"] for r in results], 'd-.', label='Trapz Log-Log')

plt.xlabel("Nθ (risoluzione angolare)")
plt.ylabel("Errore assoluto |I - I_ref|")
plt.title("Convergenza dei metodi 2D vs riferimento")
plt.grid(True, which='both', ls='--', lw=0.6)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10,6))
plt.loglog(Ns, [r["t_trap"] for r in results], 'o-', label='Trapz')
plt.loglog(Ns, [r["t_trap_numba"] for r in results], 's--', label='Trapz (Numba)')
plt.loglog(Ns, [r["t_simp"] for r in results], 'o-', label='Simpson')
plt.loglog(Ns, [r["t_simp_numba"] for r in results], 's--', label='Simpson (Numba)')
plt.loglog(Ns, [r["t_loglog"] for r in results], 'd-.', label='Trapz Log-Log')

plt.xlabel("Nθ (risoluzione)")
plt.ylabel("Tempo di esecuzione [s]")
plt.title("Scalabilità in tempo dei metodi di integrazione 2D")
plt.grid(True, which='both', ls='--', lw=0.6)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:



def E_theta_gaussian(Eiso_zero,theta, thetaw_deg=20.0):
        
        thetacore=4.0
        E = np.zeros_like(theta)
        inside = theta <= thetaw_deg
      
        E[inside]= Eiso_zero * np.exp(-(theta[inside]**2) / (2 * thetacore**2))
        E = np.maximum(E, 1e-40) #floor to avoid numerical issues
        return E 
      
def E_theta_powerlaw(Eiso_zero,theta,thetaw_deg=20.0,b=4.5):
        
        thetacore=4.0
        Eiso_zero=Eiso_zero
        
        E = np.zeros_like(theta)
        inside = theta <= thetaw_deg
        E[inside] =Eiso_zero * (1 + (theta[inside]**2) / (b * thetacore**2))**(-b/2)
        E = np.maximum(E, 1e-40) #floor to avoid numerical issues
        return E 

def compute_dynamics(Eiso,avtime, theta,scenario='ISM', energy_profile='gaussian'):
        """Calcola Lorentz factor, raggio e energia"""
        theta = np.atleast_1d(theta)
        
        if (scenario == 'ISM'):
                    depthpar = 9. / 1.
                    if energy_profile=='gaussian':
                        Energy = E_theta_gaussian(Eiso,theta)
                    elif energy_profile == 'powerlaw':
                        Energy = E_theta_powerlaw(Eiso,theta)
                    else:
                        raise ValueError(f"Unknown energy profile: {energy_profile}")  
            
        
        Gamma = (3.0 * Energy / (8.0 * np.pi * 512.0 * mpc2_erg * (c * avtime) ** 3.0)) ** 0.125
        
        Gamma = np.maximum(Gamma, 1.0)  # floor to avoid numerical issues
        R = 8. * c * avtime * Gamma ** 2

        return Gamma, R, Energy

def gamma_theta(theta, Gamma_c, theta_c):
    #theta=np.deg2rad(theta)
    #theta_c=np.deg2rad(theta_c)
    thetamin=0*u.deg
    thetamax=90*u.deg
    if np.any(theta <thetamin) or np.any(theta > thetamax):
        raise ValueError("theta deve essere compreso tra 0° e 90°")
    
    Gamma=Gamma_c * np.exp(-theta.value**2 / (2 * theta_c**2))
    Gamma_safe = np.maximum(Gamma, 1) 
    return Gamma_safe


def doppler_factor2(Gamma, theta,theta_obs, phi):

    cos_psi = np.cos(theta) * np.cos(theta_obs) + np.sin(theta) * np.sin(theta_obs) * np.cos(phi)
    beta = np.sqrt(1.0 - 1.0 / Gamma**2)
    return 1.0 / (Gamma * (1.0 - beta * cos_psi ))
    

In [ ]:
import numpy as np
from scipy.optimize import brentq
from scipy.integrate import simps

c = 2.99792458e10  # cm/s
mpc2_erg = 1.0     # <-- metti qui il tuo valore corretto

# --- le tue funzioni ---
def gammaval(Eiso, density, avtime, scenario='ISM'):
    if scenario == 'ISM':
        gamma = (1. / 8.) ** (3. / 8.) * (3.0 * Eiso / (4.0 * np.pi * density * mpc2_erg * ((c * avtime) ** 3.0))) ** 0.125
        radius = 8. * c * avtime * gamma**2
    else:
        raise ValueError("Scenario non riconosciuto")
    return gamma, radius

def doppler(gamma, theta_rad):
    if gamma <= 1.0:
        return 0.0
    beta = np.sqrt(1.0 - 1.0/gamma**2)
    return 1.0 / (gamma * (1.0 - beta*np.cos(theta_rad)))

# --- equazione per trovare il tempo di emissione ---
def emission_time_for_theta(t_obs, theta, Eiso, density, z, tmin=1e-3, tmax=1e7):
    def f(t):
        gamma, R = gammaval(Eiso, density, t)
        return (1+z)*(t - R*np.cos(theta)/c) - t_obs
    try:
        return brentq(f, tmin, tmax)
    except ValueError:
        return None

def flux_at_tobs(t_obs, Eiso, density, z=0.0, ntheta=200):
    thetas = np.linspace(0, np.pi/2, ntheta)
    integrand = []
    for th in thetas:
        t_em = emission_time_for_theta(t_obs, th, Eiso, density, z)
        if t_em is None:
            integrand.append(0.0)
            continue
        gamma, R = gammaval(Eiso, density, t_em)
        if gamma <= 1.0:
            integrand.append(0.0)
            continue
        D = doppler(gamma, th)
        contrib = 1.0   # placeholder
        integrand.append((D**2) * contrib * np.sin(th))
    return 2*np.pi*simps(integrand, thetas)

# --- esempio ---
Eiso = 1e53     # erg
density = 1.0   # cm^-3
z = 1.0
t_obs = 1e4     # s

F_val = flux_at_tobs(t_obs, Eiso, density, z=z, ntheta=200)
print("F(t_obs) =", F_val)


# --- intervallo t_obs realistico ---
t_obs_array = np.linspace(100, 1000, 50)  # s
F_vals = []

for t_obs in t_obs_array:
    F = flux_at_tobs(t_obs, Eiso, density, z=z, ntheta=200)
    F_vals.append(F)

F_vals = np.array(F_vals)

# filtra eventuali zeri
mask = F_vals > 0

# --- plot lineare ---
plt.figure(figsize=(8,4))
plt.plot(t_obs_array[mask], F_vals[mask], lw=2)
plt.xlabel(r'$t_{\rm obs}$ [s]')
plt.ylabel('Flusso (unità arbitrarie)')
plt.title('Curva di luce F(t_obs) integrata su θ')
plt.grid(True)
plt.show()



In [ ]:
# Parametri
Eiso = 1e53
density = 1.0
z = 1.0
t_em = 60  # tempo di emissione fissato (s)

# Calcolo gamma e raggio
gamma, R = gammaval(Eiso, density, t_em)

thetas = np.linspace(0, np.pi/2, 200)
t_obs_vals = (1+z)*(t_em - R*np.cos(thetas)/c)

plt.figure(figsize=(6,4))
plt.plot(thetas*180/np.pi, t_obs_vals)
plt.xlabel(r"$\theta$ [deg]")
plt.ylabel(r"$t_{\rm obs}(\theta)$ [s]")
plt.title(f"Tempo osservato per t_em={t_em:.1e} s")
plt.grid()
plt.show()

In [ ]:
t_obs = 5e3  # tempo osservato fisso
thetas = np.linspace(0, np.pi/2, 100)

t_em_vals = []
for th in thetas:
    t_em = emission_time_for_theta(t_obs, th, Eiso, density, z)
    if t_em is None:
        t_em_vals.append(np.nan)
    else:
        t_em_vals.append(t_em)

plt.figure(figsize=(6,4))
plt.plot(thetas*180/np.pi, t_em_vals)
plt.xlabel(r"$\theta$ [deg]")
plt.ylabel(r"$t_{\rm em}(\theta)$ [s]")
plt.title(f"Tempo di emissione che produce t_obs={t_obs:.1e} s")
plt.grid()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parametri
Eiso = 1e53
density = 1.0
z = 1.0

# Intervallo tempo emissione
t_lab = np.linspace(10, 1000, 200)  # tempo nel frame GRB [s]
t_obs_array = []
F_obs = []

for t in t_lab:
    gamma, R = gammaval(Eiso, density, t)
    if gamma <= 1.0:
        F_obs.append(0.0)
        t_obs_array.append((1+z)*t)
        continue

    # tempo osservato per theta=0
    t_obs = (1+z)*(t - R/c)
    t_obs_array.append(t_obs)

    # flusso semplificato
    D = doppler(gamma, 0.0)  # theta=0
    F_obs.append(D**2)       # placeholder per emissività

# Converti in array
t_obs_array = np.array(t_obs_array)
F_obs = np.array(F_obs)

# Filtra eventuali zero per log-log
mask = F_obs > 0

# --- plot ---
plt.figure(figsize=(7,4))
plt.plot(t_obs_array[mask], F_obs[mask], lw=2)
plt.xlabel(r'$t_{\rm obs}$ [s]')
plt.ylabel('Flusso (unità arbitrarie)')
plt.title('Curva di luce (tempo di emissione 10-1000 s)')
plt.grid(True, which='both', ls='--')
plt.show()
